# Question Data from Metaculus API (v4 - With Authentication)

**Date:** 2026-02-11  
**Version:** 010c - Added bot authentication for personal forecasts/scores  
**Input:** `products/Run_Question_Map_2026-02-10_v01.csv`  
**Output:** `products/Question_Data_from_API_2026-02-11_vNN.csv`

**New in this version:**
- ✅ Authenticates with METACULUS_BOT_API_TOKEN
- ✅ Fetches bot's personal forecasts and scores
- ✅ Extracts: my_latest_forecast, my_coverage, my_score
- ✅ Falls back gracefully if no auth token provided

**Data extracted:**
- Community scores (from `question.aggregations.unweighted.score_data`)
- Bot's personal forecasts and scores (requires auth)

In [1]:
# Imports
import requests
import pandas as pd
import time
import json
import os
from pathlib import Path
from datetime import date
from typing import Dict, Optional

print("✅ Imports successful")

✅ Imports successful


In [2]:
# Configuration
INPUT_FILE = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products/Run_Question_Map_2026-02-10_v01.csv")
OUTPUT_DIR = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products")
API_BASE = "https://www.metaculus.com/api2/questions"
TEST_LIMIT = 5  # Set to 5 for testing, None for all questions

# Authentication - will use environment variable or set manually here
METACULUS_TOKEN = os.getenv('METACULUS_BOT_API_TOKEN')  # or set directly: METACULUS_TOKEN = 'your_token_here'

# Rate limiting settings
RATE_LIMIT_DELAY = 2.0  # Seconds between requests
MAX_RETRIES = 3  # Max retry attempts on 429 error
BACKOFF_BASE = 5  # Base delay for exponential backoff (seconds)
PROGRESS_SAVE_INTERVAL = 25  # Save progress every N questions

print(f"Input file: {INPUT_FILE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Test limit: {TEST_LIMIT}")
print(f"Authentication: {'✅ Token found' if METACULUS_TOKEN else '❌ No token - will fetch public data only'}")
print(f"Rate limit: {RATE_LIMIT_DELAY}s between requests")

Input file: C:\Users\Donni\projects\metac_bot_Spring_2026\products\Run_Question_Map_2026-02-10_v01.csv
Output dir: C:\Users\Donni\projects\metac_bot_Spring_2026\products
Test limit: 5
Authentication: ❌ No token - will fetch public data only
Rate limit: 2.0s between requests


In [3]:
# Load question list
df_runs = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df_runs)} run records")

# Extract unique question numbers (filter out empty values)
question_numbers = df_runs['question_number'].dropna().astype(int).unique().tolist()
question_numbers.sort()

print(f"Found {len(question_numbers)} unique questions")
print(f"Range: {min(question_numbers)} to {max(question_numbers)}")
print(f"First 10: {question_numbers[:10]}")

Loaded 1260 run records
Found 193 unique questions
Range: 41379 to 42078
First 10: [41379, 41380, 41382, 41383, 41384, 41385, 41386, 41389, 41392, 41396]


In [4]:
# API fetch function with authentication
def fetch_question_data(question_id: int, max_retries: int = MAX_RETRIES) -> Optional[Dict]:
    """Fetch question data from Metaculus API with optional authentication."""
    url = f"{API_BASE}/{question_id}/"
    
    # Add auth header if token available
    headers = {}
    if METACULUS_TOKEN:
        headers['Authorization'] = f'Token {METACULUS_TOKEN}'
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=headers, timeout=30)
            
            # Success
            if response.status_code == 200:
                return response.json()
            
            # Rate limit - retry with exponential backoff
            if response.status_code == 429:
                wait_time = BACKOFF_BASE * (2 ** attempt)
                print(f"\n  ⏳ Rate limited, waiting {wait_time}s... ", end='')
                time.sleep(wait_time)
                continue
            
            # Auth error
            if response.status_code == 401:
                print(f"\n  ⚠️  Authentication failed - token may be invalid")
                return None
            
            # Other error
            response.raise_for_status()
            
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                print(f"\n  ⚠️  API error after {max_retries} attempts: {e}")
                return None
            time.sleep(2)
    
    return None

print("✅ fetch_question_data() with auth support defined")

✅ fetch_question_data() with auth support defined


In [5]:
# Data extraction function with bot's personal data
def extract_question_fields(data: Dict) -> Dict:
    """Extract relevant fields from API response including personal forecasts."""
    
    # Top-level fields
    result = {
        'question_id': data.get('id'),
        'title': data.get('title', ''),
        'short_title': data.get('short_title', ''),
        'slug': data.get('slug', ''),
        'status': data.get('status', ''),
        'resolved': data.get('resolved', False),
        'comment_count': data.get('comment_count', 0),
        'nr_forecasters': data.get('nr_forecasters', 0),
        'forecasts_count': data.get('forecasts_count', 0),
        'author_username': data.get('author_username', ''),
        'curation_status': data.get('curation_status', ''),
    }
    
    # Dates
    result['created_at'] = data.get('created_at', '')
    result['published_at'] = data.get('published_at', '')
    result['edited_at'] = data.get('edited_at', '')
    result['open_time'] = data.get('open_time', '')
    result['actual_close_time'] = data.get('actual_close_time', '')
    result['scheduled_close_time'] = data.get('scheduled_close_time', '')
    result['actual_resolve_time'] = data.get('actual_resolve_time', '')
    result['scheduled_resolve_time'] = data.get('scheduled_resolve_time', '')
    
    # Question sub-object
    question = data.get('question', {})
    result['question_type'] = question.get('type', '')
    result['resolution'] = question.get('resolution')
    result['resolution_set_time'] = question.get('resolution_set_time', '')
    result['question_weight'] = question.get('question_weight', '')
    result['description'] = question.get('description', '')
    result['resolution_criteria'] = question.get('resolution_criteria', '')
    result['fine_print'] = question.get('fine_print', '')
    
    # Question-type specific fields
    if result['question_type'] == 'multiple_choice':
        result['mc_options'] = json.dumps(question.get('options', []))
    else:
        result['mc_options'] = ''
    
    if result['question_type'] == 'numeric':
        scaling = question.get('scaling', {})
        result['numeric_range_min'] = scaling.get('range_min', '')
        result['numeric_range_max'] = scaling.get('range_max', '')
        result['open_upper_bound'] = scaling.get('open_upper_bound', '')
        result['open_lower_bound'] = scaling.get('open_lower_bound', '')
    else:
        result['numeric_range_min'] = ''
        result['numeric_range_max'] = ''
        result['open_upper_bound'] = ''
        result['open_lower_bound'] = ''
    
    # Tournament info
    projects = data.get('projects', {})
    default_project = projects.get('default_project', {})
    result['tournament_id'] = default_project.get('id', '')
    result['tournament_name'] = default_project.get('name', '')
    result['tournament_slug'] = default_project.get('slug', '')
    
    # Get aggregations from question.aggregations
    question_aggregations = question.get('aggregations', {})
    unweighted = question_aggregations.get('unweighted', {})
    latest = unweighted.get('latest', {})
    
    # Community forecast from latest
    result['community_forecaster_count'] = latest.get('forecaster_count', '')
    forecast_values = latest.get('forecast_values', [])
    
    # Format community forecast based on type
    if result['question_type'] == 'binary' and len(forecast_values) == 2:
        result['community_forecast'] = f"{forecast_values[1]:.1%}"  # p_yes
        result['community_forecast_mean'] = forecast_values[1]
    elif result['question_type'] == 'multiple_choice':
        result['community_forecast'] = json.dumps(forecast_values)
        result['community_forecast_mean'] = ''
    elif result['question_type'] == 'numeric':
        means = latest.get('means', [])
        if means:
            result['community_forecast_mean'] = means[0]
            result['community_forecast'] = f"{means[0]:.2f}"
        else:
            result['community_forecast'] = ''
            result['community_forecast_mean'] = ''
    else:
        result['community_forecast'] = ''
        result['community_forecast_mean'] = ''
    
    # Interval bounds from latest
    interval_lower = latest.get('interval_lower_bounds', [])
    interval_upper = latest.get('interval_upper_bounds', [])
    result['community_interval_lower'] = interval_lower[0] if interval_lower else ''
    result['community_interval_upper'] = interval_upper[0] if interval_upper else ''
    
    # Community score data from question.aggregations.unweighted.score_data
    score_data = unweighted.get('score_data', {})
    result['community_coverage'] = score_data.get('coverage', '')
    result['community_peer_score'] = score_data.get('peer_score', '')
    result['community_baseline_score'] = score_data.get('baseline_score', '')
    result['community_spot_peer_score'] = score_data.get('spot_peer_score', '')
    result['community_spot_baseline_score'] = score_data.get('spot_baseline_score', '')
    
    # Bot's personal forecast and score (if authenticated)
    my_forecast = question.get('my_forecasts', {})
    if my_forecast:
        latest_forecast = my_forecast.get('latest', {})
        result['my_latest_forecast'] = json.dumps(latest_forecast.get('forecast_values', []))
        result['my_latest_forecast_time'] = latest_forecast.get('t', '')
        
        # My score data
        my_score_data = my_forecast.get('score_data', {})
        result['my_coverage'] = my_score_data.get('coverage', '')
        result['my_peer_score'] = my_score_data.get('peer_score', '')
        result['my_baseline_score'] = my_score_data.get('baseline_score', '')
        result['my_spot_peer_score'] = my_score_data.get('spot_peer_score', '')
        result['my_spot_baseline_score'] = my_score_data.get('spot_baseline_score', '')
    else:
        result['my_latest_forecast'] = ''
        result['my_latest_forecast_time'] = ''
        result['my_coverage'] = ''
        result['my_peer_score'] = ''
        result['my_baseline_score'] = ''
        result['my_spot_peer_score'] = ''
        result['my_spot_baseline_score'] = ''
    
    return result

print("✅ extract_question_fields() with personal forecasts defined")

✅ extract_question_fields() with personal forecasts defined


In [6]:
# Helper functions
def get_next_version_file(base_name: str) -> Path:
    """Get next available version number for output file."""
    version = 1
    while True:
        filename = OUTPUT_DIR / f"{base_name}_v{version:02d}.csv"
        if not filename.exists():
            return filename
        version += 1

def load_existing_results(base_name: str) -> pd.DataFrame:
    """Load most recent output file if it exists."""
    import glob
    pattern = str(OUTPUT_DIR / f"{base_name}_v*.csv")
    files = sorted(glob.glob(pattern))
    if files:
        print(f"Found existing file: {Path(files[-1]).name}")
        return pd.read_csv(files[-1])
    return pd.DataFrame()

print("✅ Helper functions defined")

✅ Helper functions defined


In [7]:
# Test on single question
test_id = question_numbers[0]
print(f"Testing API fetch for question {test_id}...")

test_data = fetch_question_data(test_id)
if test_data:
    print(f"✅ API response received ({len(test_data)} top-level keys)")
    test_fields = extract_question_fields(test_data)
    print(f"✅ Extracted {len(test_fields)} fields")
    
    print("\nBasic info:")
    for k in ['question_id', 'title', 'question_type', 'status', 'resolved', 'resolution']:
        print(f"  {k}: {test_fields.get(k)}")
    
    print("\nCommunity scores:")
    for k in ['community_coverage', 'community_peer_score', 'community_baseline_score']:
        print(f"  {k}: {test_fields.get(k)}")
    
    print("\nBot's personal data:")
    for k in ['my_latest_forecast', 'my_coverage', 'my_peer_score', 'my_baseline_score']:
        val = test_fields.get(k)
        if val:
            print(f"  {k}: {val}")
        else:
            print(f"  {k}: (empty - auth may be needed)")
else:
    print("❌ API fetch failed")

Testing API fetch for question 41379...
✅ API response received (31 top-level keys)
✅ Extracted 51 fields

Basic info:
  question_id: 41379
  title: Will the interest in “el salvador” change between 2026-01-05 and 2026-01-17 according to Google Trends?
  question_type: multiple_choice
  status: resolved
  resolved: True
  resolution: Doesn't change

Community scores:
  community_coverage: 0.9951493148008984
  community_peer_score: 11.442805675138914
  community_baseline_score: 31.101758399468846

Bot's personal data:
  my_latest_forecast: (empty - auth may be needed)
  my_coverage: (empty - auth may be needed)
  my_peer_score: (empty - auth may be needed)
  my_baseline_score: (empty - auth may be needed)


In [8]:
# Batch fetch
questions_to_fetch = question_numbers[:TEST_LIMIT] if TEST_LIMIT else question_numbers

base_name = f"Question_Data_from_API_{date.today()}"
existing_df = load_existing_results(base_name)
already_fetched = set(existing_df['question_id'].tolist()) if not existing_df.empty else set()

if already_fetched:
    print(f"\nResuming: {len(already_fetched)} questions already fetched")
    questions_to_fetch = [q for q in questions_to_fetch if q not in already_fetched]
    print(f"Remaining: {len(questions_to_fetch)} questions\n")
else:
    print(f"\nFetching {len(questions_to_fetch)} questions...\n")

results = existing_df.to_dict('records') if not existing_df.empty else []
failed = []
last_save_count = len(results)

for i, qnum in enumerate(questions_to_fetch, 1):
    print(f"[{i}/{len(questions_to_fetch)}] Q{qnum}...", end='')
    
    data = fetch_question_data(qnum)
    if data:
        try:
            fields = extract_question_fields(data)
            results.append(fields)
            has_my_forecast = '✓' if fields.get('my_latest_forecast') else '○'
            print(f" ✓ ({fields['question_type']}, {fields['status']}, my:{has_my_forecast})")
        except Exception as e:
            print(f" ⚠️  Extract failed: {e}")
            failed.append(qnum)
    else:
        print(f" ❌ Fetch failed")
        failed.append(qnum)
    
    # Periodic progress save
    if (len(results) - last_save_count) >= PROGRESS_SAVE_INTERVAL:
        temp_df = pd.DataFrame(results)
        progress_file = OUTPUT_DIR / f"{base_name}_progress.csv"
        temp_df.to_csv(progress_file, index=False)
        print(f"  💾 Progress saved: {len(results)} questions")
        last_save_count = len(results)
    
    # Rate limiting
    if i < len(questions_to_fetch):
        time.sleep(RATE_LIMIT_DELAY)

print(f"\n✅ Successfully fetched {len(results)} total questions")
if failed:
    print(f"❌ Failed to fetch {len(failed)} questions: {failed}")

Found existing file: Question_Data_from_API_2026-02-11_v01.csv

Resuming: 192 questions already fetched
Remaining: 0 questions


✅ Successfully fetched 192 total questions


In [9]:
# Convert to DataFrame
df = pd.DataFrame(results)
print(f"DataFrame shape: {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")

DataFrame shape: (192, 44)
Columns (44): ['question_id', 'title', 'short_title', 'slug', 'status', 'resolved', 'comment_count', 'nr_forecasters', 'forecasts_count', 'author_username', 'curation_status', 'created_at', 'published_at', 'edited_at', 'open_time', 'actual_close_time', 'scheduled_close_time', 'actual_resolve_time', 'scheduled_resolve_time', 'question_type', 'resolution', 'resolution_set_time', 'question_weight', 'description', 'resolution_criteria', 'fine_print', 'mc_options', 'numeric_range_min', 'numeric_range_max', 'open_upper_bound', 'open_lower_bound', 'tournament_id', 'tournament_name', 'tournament_slug', 'community_forecaster_count', 'community_forecast', 'community_forecast_mean', 'community_interval_lower', 'community_interval_upper', 'coverage', 'peer_score', 'baseline_score', 'spot_peer_score', 'spot_baseline_score']


In [10]:
# Data quality checks
print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

print(f"\nQuestion Types:")
print(df['question_type'].value_counts())

print(f"\nStatus:")
print(df['status'].value_counts())

print(f"\nCommunity Score data:")
print(f"  With community_coverage: {(df['community_coverage'] != '').sum()}")
print(f"  With community_peer_score: {(df['community_peer_score'] != '').sum()}")

print(f"\nBot's Personal Data:")
print(f"  With my_latest_forecast: {(df['my_latest_forecast'] != '').sum()}")
print(f"  With my_coverage: {(df['my_coverage'] != '').sum()}")
print(f"  With my_peer_score: {(df['my_peer_score'] != '').sum()}")

if (df['my_latest_forecast'] != '').sum() == 0:
    print("\n⚠️  No personal forecasts found - check if METACULUS_BOT_API_TOKEN is valid")

print(f"\nTournaments:")
print(df['tournament_name'].value_counts())

DATA QUALITY CHECKS

Question Types:
question_type
binary             91
numeric            59
multiple_choice    36
discrete            6
Name: count, dtype: int64

Status:
status
resolved    101
closed       91
Name: count, dtype: int64

Community Score data:


KeyError: 'community_coverage'

In [ ]:
# Save to VERSIONED CSV
output_file = get_next_version_file(base_name)
df.to_csv(output_file, index=False)
print(f"\n✅ Saved to: {output_file.name}")
print(f"   Rows: {len(df)}")
print(f"   Columns: {len(df.columns)}")
print(f"   Size: {output_file.stat().st_size / 1024:.1f} KB")

# Clean up progress file
progress_file = OUTPUT_DIR / f"{base_name}_progress.csv"
if progress_file.exists():
    progress_file.unlink()
    print(f"   Cleaned up progress file")

In [ ]:
# Display sample with both community and personal data
print("\n" + "=" * 100)
print("SAMPLE DATA (first 5 questions)")
print("=" * 100)

display_cols = ['question_id', 'short_title', 'question_type', 'resolved', 
                'community_coverage', 'community_peer_score', 
                'my_coverage', 'my_peer_score']
available_cols = [c for c in display_cols if c in df.columns]
df[available_cols].head(5)